In [1]:
import pandas as pd
import openpyxl
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from time import sleep
import pandas as pd
import os
import time
import shutil

In [2]:
df=pd.read_excel("Hines-Conservice-343821.xlsx")
wb = openpyxl.load_workbook('Hines-Conservice-343821.xlsx')


In [3]:
sheets = wb.sheetnames
ws = wb[sheets[0]]
links=[]

for i in range(0,len(df)):
    try:
        link=str(ws.cell(row=i+2, column=7).hyperlink.target)
        #print(link)
        links.append(link)
    except:
        print("No Link found in entry #"+str(i))
        links.append("")

In [4]:
df["links"]=links
df=df.reset_index(drop=True)

In [5]:
temp_path="C:\\Users\\Minto Mathew\\Desktop\\Bill Download\\Bills\\Temp" #Folder path of where the file has to be downloaded with leading double slashes(\\)
download_path="C:\\Users\\Minto Mathew\\Desktop\\Bill Download\\Bills"

#Function to wait until the present download is completed
def download_wait(path_to_downloads):
    seconds = 0
    dl_wait = True
    while dl_wait and seconds < 60:
        time.sleep(1)
        dl_wait = False
        for fname in os.listdir(path_to_downloads):
            if fname.endswith('.crdownload'):
                dl_wait = True
        seconds += 1
    return seconds

In [6]:
chromeOptions=Options()
#Setting the default pdf action as download
chromeOptions.add_experimental_option("prefs", {"download.default_directory":temp_path ,
"plugins.always_open_pdf_externally": True,
  "download.prompt_for_download": False,
  "download.directory_upgrade": True,
  "safebrowsing.enabled": True})

In [7]:
#Function to download the pdf
def pdf_download(filename,link):
    
    driver = webdriver.Chrome(options=chromeOptions)
    try:
        driver.get(link)
        driver.maximize_window() # For maximizing window
        driver.implicitly_wait(3)
        iframe = driver.find_element(By.TAG_NAME, "iframe")
        driver.switch_to.frame(iframe)
        open_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Open')]")  # Adjust if necessary
        open_button.click()
        
        time.sleep(5)
        #seconds = 0
        dl_wait = True
        #while dl_wait and seconds < 600:
        while dl_wait:
            time.sleep(1)
            dl_wait = False
            for fname in os.listdir(temp_path):
                if fname.endswith('.crdownload'):
                    dl_wait = True
            #seconds += 1
        
        driver.close()
    except:
        print("No Bill Found\n")
        driver.close()
        return
    
    files = [f for f in os.listdir(temp_path) if f.endswith('.pdf')]
    if files:
        latest_file = max(files, key=lambda f: os.path.getctime(os.path.join(temp_path, f)))  # Get the most recent file
        print(f"Downloaded file: {latest_file}")

        # Define the new file name
        new_file_name = filename + ".pdf"  # You can customize this name as needed
        new_file_path = os.path.join(temp_path, new_file_name)

        # Rename the file
        os.rename(os.path.join(temp_path, latest_file), new_file_path)
        print(f"File renamed to: {new_file_name}\n")
        
        #shutil.copy(new_file_name, download_path)
        #os.remove(new_file_name)
    else:
        print("No PDF file downloaded.")
    

os.chdir(temp_path)
for i in range(266,len(df)):
    print("Item #"+str(i))
    try:
        pdf_download(df['Filename'][i],df['links'][i])
    except NoSuchElementException:
        print("No Bill Found")

In [8]:
os.chdir(temp_path)
for i in range(len(df)):
    print("Item #"+str(i))
    try:
        pdf_download(df['File Name'][i],df['links'][i])
    except:
        print('Error - iteration {}'.format(i))
        continue

Item #0
Downloaded file: 251159179039090226201116002250199222023118210033.pdf
File renamed to: WE Energies0740116191-00337AJK25061424371ElectricityHouse.pdf

Item #1
Downloaded file: 006149088103236112254111143106151221142004021147.pdf
File renamed to: WE Energies0740116191-00345HIX25061710092ElectricityHouse.pdf

Item #2
Downloaded file: 143087114151156170170244027242000019106227017105.pdf
File renamed to: WE Energies0740116191-00680HIX25061709876GasHouse.pdf

Item #3
Downloaded file: 099024165100242054030191044089134125170147060115.pdf
File renamed to: PG&E5506928808-6AJC25060517533ElectricityHouse.pdf

Item #4
Downloaded file: 245010013176111130180007115065237139109166186039.pdf
File renamed to: PG&E8013923549-5AJC25060519620ElectricityHouse.pdf

Item #5
Downloaded file: 062040190027045016146219197083215107182019118033.pdf
File renamed to: PG&E9294572370-5PSA25060904598GasHouse.pdf

Item #6
Downloaded file: 017103168241233038222201172175187007051108143203.pdf
File renamed to: DC Wat

os.chdir(temp_path)
pdf_download(df['Filename'][64],df['links'][64])

pdf_download(df['File Name'][15],df['links'][15])

for i in [445,446,447,448,287,158,147,148]:
    print("Item #"+str(i))
    try:
        pdf_download(df['File Name'][i],df['links'][i])
    except:
        print('Error - iteration {}'.format(i))
        continue